In [63]:
import torch


# 将名字序列转换为 (上下文 -> 下一个字符) 的监督学习样本
# - word_list: 单词列表
# - stoi: 字符到索引的映射（约定 '.' 是 0）
# - block_size: 上下文窗口长度
def build_dataset(word_list, stoi, block_size):
    X_data = []
    Y_data = []

    for word in word_list:
        # 初始上下文全 0，表示全是结束符占位
        context = [0] * block_size
        for ch in word + ".":
            idx = stoi[ch]
            X_data.append(context)
            Y_data.append(idx)
            # 左移窗口并拼接当前字符索引
            context = context[1:] + [idx]

    X = torch.tensor(X_data, dtype=torch.long)
    Y = torch.tensor(Y_data, dtype=torch.long)
    return X, Y


# 全连接层：out = x @ W + b
class Linear:
    def __init__(self, fan_in, fan_out, bias=True, generator=None):
        self.weight = torch.randn((fan_in, fan_out), generator=generator) / fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self, x):
        out = x @ self.weight
        if self.bias is not None:
            out = out + self.bias
        self.out = out
        return self.out

    # 返回可训练参数，便于统一优化
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])


# 按“特征维”归一化：同一列特征在 batch 内做标准化
class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True

        # 可训练缩放/平移参数
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

        # 推理阶段使用的滑动统计量
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)

    def __call__(self, x):
        if self.training:
            xmean = x.mean(dim=0, keepdim=True)
            xvar = x.var(dim=0, keepdim=True, unbiased=False)
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean.squeeze(0)
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar.squeeze(0)
        else:
            xmean = self.running_mean
            xvar = self.running_var

        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]


# 基于 BatchNorm 的思路实现 LayerNorm：
# 仍然是标准化 + gamma/beta，但统计维度改为“每个样本自身的特征维”
class LayerNorm1d:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.training = True

        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        xmean = x.mean(dim=1, keepdim=True)
        xvar = x.var(dim=1, keepdim=True, unbiased=False)
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]


# 非线性激活层
class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out

    def parameters(self):
        return []


# 嵌入层：用索引查表得到向量
class Embedding:
    def __init__(self, num_embeddings, embedding_dim, generator=None):
        self.weight = torch.randn((num_embeddings, embedding_dim), generator=generator)

    def __call__(self, indices):
        self.out = self.weight[indices]
        return self.out

    def parameters(self):
        return [self.weight]


# 展平层：把高维特征拼成二维 (B, -1)
class Flatten:
    def __call__(self, x):
        self.out = x.view(x.shape[0], -1)
        return self.out

    def parameters(self):
        return []


# 顺序容器：按层依次前向，并汇总所有参数
class Sequential:
    def __init__(self, layers):
        self.layers = layers

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        self.out = x
        return self.out

    def parameters(self):
        return [param for layer in self.layers for param in layer.parameters()]

    # 切换到训练模式（影响有 training 属性的层）
    def train(self):
        for layer in self.layers:
            if hasattr(layer, "training"):
                layer.training = True

    # 切换到评估模式
    def eval(self):
        for layer in self.layers:
            if hasattr(layer, "training"):
                layer.training = False


# 测试辅助：判断两个张量是否近似相等
def assert_close(a, b, tol=1e-5, msg=""):
    ok = torch.allclose(a, b, atol=tol, rtol=tol)
    if not ok:
        raise AssertionError(msg or f"Tensor not close:\n{a}\n!=\n{b}")


In [64]:
# 1) build_dataset 教学用例：展示“滑动上下文 -> 下一个字符”
stoi = {".": 0, "a": 1, "b": 2, "c": 3}
itos = {i: ch for ch, i in stoi.items()}
words = ["ab", "ac"]
block_size = 4

X, Y = build_dataset(words, stoi=stoi, block_size=block_size)

print("输入单词:", words)
print("X shape:", X.shape, "Y shape:", Y.shape)
print("\n前 6 条训练样本（上下文 -> 目标）:")
for i in range(6):
    context = "".join(itos[idx] for idx in X[i].tolist())
    target = itos[Y[i].item()]
    print(f"{i:>2}: '{context}' -> '{target}'")

# 关键性质：每个长度为 L 的词会产生 L+1 条样本（包含结束符）
expected_num = sum(len(word) + 1 for word in words)
assert X.shape == (expected_num, block_size)
assert Y.shape == (expected_num,)
print("\n✅ build_dataset: 成功把字符序列转换为监督学习样本")

输入单词: ['ab', 'ac']
X shape: torch.Size([6, 4]) Y shape: torch.Size([6])

前 6 条训练样本（上下文 -> 目标）:
 0: '....' -> 'a'
 1: '...a' -> 'b'
 2: '..ab' -> '.'
 3: '....' -> 'a'
 4: '...a' -> 'c'
 5: '..ac' -> '.'

✅ build_dataset: 成功把字符序列转换为监督学习样本


In [65]:
# 2) Linear 教学用例：仿射变换 y = xW + b
lin = Linear(3, 2, bias=True)

# 固定参数，便于手算对照
lin.weight = torch.tensor([
    [1.0, 2.0],
    [0.0, 1.0],
    [1.0, -1.0],
])
lin.bias = torch.tensor([0.5, -0.5])

x = torch.tensor([
    [1.0, 2.0, 3.0],
    [0.0, 1.0, 0.0],
])
out = lin(x)
manual = x @ lin.weight + lin.bias

print("x shape:", x.shape)
print("weight shape:", lin.weight.shape)
print("out shape:", out.shape)
print("\nLinear 输出:\n", out)
print("\n手工计算输出:\n", manual)

assert_close(out, manual, msg="Linear 应等于 x @ W + b")
assert len(lin.parameters()) == 2
print("\n✅ Linear: 展示了维度变换与参数作用")

x shape: torch.Size([2, 3])
weight shape: torch.Size([3, 2])
out shape: torch.Size([2, 2])

Linear 输出:
 tensor([[4.5000, 0.5000],
        [0.5000, 0.5000]])

手工计算输出:
 tensor([[4.5000, 0.5000],
        [0.5000, 0.5000]])

✅ Linear: 展示了维度变换与参数作用


In [66]:
# 3) BatchNorm1d 教学用例：按“特征列”在 batch 内归一化
# batchnorm和layernorm传入的参数都是特征维度
bn = BatchNorm1d(dim=3)
# 4 个样本，3 个特征维度
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [2.0, 3.0, 4.0],
    [3.0, 4.0, 5.0],
    [4.0, 5.0, 6.0],
])

# 训练模式：使用当前 batch 的统计量
bn.training = True
out_train = bn(x)
print("输入 x:\n", x)
print("输出 out:\n", out_train)


输入 x:
 tensor([[1., 2., 3.],
        [2., 3., 4.],
        [3., 4., 5.],
        [4., 5., 6.]])
输出 out:
 tensor([[-1.3416, -1.3416, -1.3416],
        [-0.4472, -0.4472, -0.4472],
        [ 0.4472,  0.4472,  0.4472],
        [ 1.3416,  1.3416,  1.3416]])


In [67]:
# 4) LayerNorm1d 教学用例：按“样本行”归一化（与 BatchNorm 维度不同）
ln = LayerNorm1d(dim=3)
# 2 个样本，3 个特征维度
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [2.0, 4.0, 6.0],
])

out = ln(x)

print("输入 x:\n", x)
print("输出 out:\n", out)


输入 x:
 tensor([[1., 2., 3.],
        [2., 4., 6.]])
输出 out:
 tensor([[-1.2247,  0.0000,  1.2247],
        [-1.2247,  0.0000,  1.2247]])


In [68]:
# 5) Tanh 教学用例：非线性压缩到 (-1, 1)
act = Tanh()
x = torch.tensor([[-3.0, -1.0, 0.0, 1.0, 3.0]])
out = act(x)

print("输入:", x)
print("输出:", out)
print("输出范围: [", out.min().item(), ",", out.max().item(), "]")


输入: tensor([[-3., -1.,  0.,  1.,  3.]])
输出: tensor([[-0.9951, -0.7616,  0.0000,  0.7616,  0.9951]])
输出范围: [ -0.9950547814369202 , 0.9950547814369202 ]


In [69]:
# 6) Embedding 教学用例：索引查表 + 重复索引共享同一向量
emb = Embedding(num_embeddings=5, embedding_dim=2)
emb.weight = torch.tensor([
    [0.0, 0.1],
    [1.0, 1.1],
    [2.0, 2.2],
    [3.0, 3.3],
    [4.0, 4.4],
])

indices = torch.tensor([[1, 3, 1], [0, 2, 4]])
out = emb(indices)

print("indices shape:", indices.shape)
print("embedding 输出 shape:", out.shape)
print("\nlookup 结果:\n", out)


indices shape: torch.Size([2, 3])
embedding 输出 shape: torch.Size([2, 3, 2])

lookup 结果:
 tensor([[[1.0000, 1.1000],
         [3.0000, 3.3000],
         [1.0000, 1.1000]],

        [[0.0000, 0.1000],
         [2.0000, 2.2000],
         [4.0000, 4.4000]]])


In [70]:
# 7) Flatten 教学用例：仅改变形状，不改变样本内部元素顺序
flat = Flatten()
x = torch.arange(24.0).view(2, 3, 4)
out = flat(x)

print("原始 shape:", x.shape)
print("展平 shape:", out.shape)
print("\n第 1 个样本展平前:\n", x[0])
print("\n第 1 个样本展平后:\n", out[0])


原始 shape: torch.Size([2, 3, 4])
展平 shape: torch.Size([2, 12])

第 1 个样本展平前:
 tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])

第 1 个样本展平后:
 tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11.])


In [71]:
# 8) Sequential 教学用例：把多个层像流水线一样串起来
lin1 = Linear(4, 3, bias=False)
lin1.weight = torch.tensor([
    [1.0, 0.0, 0.0],
    [0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0],
    [1.0, 1.0, 1.0],
])

model = Sequential([lin1, Tanh()])

x = torch.tensor([[1.0, 2.0, 3.0, 4.0]])
out = model(x)
manual = torch.tanh(x @ lin1.weight)

print("输入:", x)
print("Sequential 输出:", out)
print("手工逐层输出:", manual)
print("参数总数:", sum(p.numel() for p in model.parameters()))


输入: tensor([[1., 2., 3., 4.]])
Sequential 输出: tensor([[0.9999, 1.0000, 1.0000]])
手工逐层输出: tensor([[0.9999, 1.0000, 1.0000]])
参数总数: 12
